# 18. RNA-seq: Measuring Which Genes Are Switched On

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `samples.py`

Turn GEO's series matrix for GSE52778 into a table of samples.


In [ ]:
# Turn GEO's series matrix for GSE52778 into a table of samples.
import gzip

import pandas as pd

fields = {}
with gzip.open("../data/GSE52778_series_matrix.txt.gz", "rt") as handle:
    for line in handle:
        if not line.startswith("!Sample_"):
            continue
        key, *values = line.rstrip("\n").split("\t")
        values = [v.strip('"') for v in values]
        if key == "!Sample_characteristics_ch1":
            key = values[0].split(":")[0]  # "treatment: Untreated"
            values = [v.split(": ", 1)[1] for v in values]
        fields.setdefault(key, values)

samples = pd.DataFrame(
    {
        "sample": fields["!Sample_geo_accession"],
        "cell_line": fields["cell line"],
        "treatment": fields["treatment"],
    }
)
samples.to_csv("samples.tsv", sep="\t", index=False)
print(samples.to_string(index=False))

## `normalize.py`

Three ways to make RNA-seq counts comparable between samples, applied to the untreated and dexamethasone treated airway cells of GSE52778.


In [ ]:
# Three ways to make RNA-seq counts comparable between samples, applied to
# the untreated and dexamethasone treated airway cells of GSE52778.
import numpy as np
import pandas as pd

counts = pd.read_csv(
    "../data/GSE52778_counts.tsv.gz", sep="\t", index_col="GeneID"
)
samples = pd.read_csv("samples.tsv", sep="\t")
samples = samples[
    samples["treatment"].isin(["Untreated", "Dexamethasone"])
]
counts = counts[samples["sample"]]
counts = counts[counts.sum(axis=1) > 0]  # drop genes no read reached
print(f"{len(counts):,} genes with any reads in {len(samples)} samples")

# Counts per million: divide by the library size
library = counts.sum()
cpm = counts / library * 1e6

# Transcripts per million: divide by transcript length first
features = pd.read_csv(
    "../data/human_feature_table.txt.gz",
    sep="\t",
    dtype={"chromosome": str},
)
rnas = features[features["# feature"].isin(["mRNA", "ncRNA", "misc_RNA"])]
length = rnas.groupby("GeneID")["product_length"].max() / 1000  # kb
with_length = counts.loc[counts.index.isin(length.index)]
per_kb = with_length.div(length[with_length.index], axis=0)
tpm = per_kb / per_kb.sum() * 1e6

# Median of ratios: compare each sample with a pseudo sample made of every
# gene's geometric mean, using only genes with reads in every sample
expressed = counts[(counts > 0).all(axis=1)]
log_mean = np.log(expressed).mean(axis=1)
ratios = np.log(expressed).sub(log_mean, axis=0)
size_factors = np.exp(ratios.median())

table = pd.DataFrame(
    {
        "treatment": samples.set_index("sample")["treatment"].map(
            {"Untreated": "none", "Dexamethasone": "dex"}
        ),
        "millions": (library / 1e6).round(1),
        "size_factor": size_factors.round(2),
        "FKBP5_cpm": cpm.loc[2289].round(0),
        "FKBP5_tpm": tpm.loc[2289].round(0),
    }
)
print(table.to_string())
cpm.to_csv("cpm.tsv.gz", sep="\t")